In [ ]:
import pandas as pd

movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
tags = pd.read_csv("tags.csv")

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
# Merge tags with movies
tags_grouped = tags.groupby("movieId")["tag"].apply(lambda x: " ".join(x)).reset_index()

movies = movies.merge(tags_grouped, on="movieId", how="left")

# Fill NaN tags
movies["tag"] = movies["tag"].fillna("")

# Create rich content
movies["content"] = movies["title"] + " " + movies["genres"] + " " + movies["tag"]

movies = movies[["movieId", "title", "content"]]

movies.head()

,movieId,title,content
0,1,Toy Story (1995),Toy Story (1995) Adventure|Animation|Children|...
1,2,Jumanji (1995),Jumanji (1995) Adventure|Children|Fantasy fant...
2,3,Grumpier Old Men (1995),Grumpier Old Men (1995) Comedy|Romance moldy old
3,4,Waiting to Exhale (1995),Waiting to Exhale (1995) Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Father of the Bride Part II (1995) Comedy preg...


In [ ]:
!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.5 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(movies['content'].tolist(), show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/305 [00:00<?, ?it/s]

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("Movies indexed:", index.ntotal)

Movies indexed: 9742


In [ ]:
# Compute average ratings
avg_ratings = ratings.groupby("movieId")["rating"].mean().reset_index()

movies = movies.merge(avg_ratings, on="movieId", how="left")
movies["rating"] = movies["rating"].fillna(0)

In [ ]:
def recommend_movies(query, top_k=5):
    query_embedding = model.encode([query])

    distances, indices = index.search(np.array(query_embedding), top_k * 3)

    candidates = movies.iloc[indices[0]].copy()

    # Boost with ratings
    candidates["score"] = (1 / (1 + distances[0])) + 0.3 * candidates["rating"]

    results = candidates.sort_values(by="score", ascending=False).head(top_k)

    return results[["title", "rating"]]

In [ ]:
def explain(query, df):
    print(f"\n🎯 Based on: '{query}'\n")

    for _, row in df.iterrows():
        print(f"🎬 {row['title']} (⭐ {round(row['rating'],2)})")
        print(f"   → Matches themes like: {query}\n")

In [ ]:
def explain(query, df):
    print(f"\n🎯 Based on: '{query}'\n")

    for _, row in df.iterrows():
        print(f"🎬 {row['title']} (⭐ {round(row['rating'],2)})")
        print(f"   → Matches themes like: {query}\n")

In [ ]:
query = "emotional sci-fi like Interstellar"

recs = recommend_movies(query)
explain(query, recs)


🎯 Based on: 'emotional sci-fi like Interstellar'

🎬 Another Earth (2011) (⭐ 4.5)
   → Matches themes like: emotional sci-fi like Interstellar

🎬 Code 46 (2003) (⭐ 4.5)
   → Matches themes like: emotional sci-fi like Interstellar

🎬 Signal, The (2014) (⭐ 4.5)
   → Matches themes like: emotional sci-fi like Interstellar

🎬 Interstellar (2014) (⭐ 3.99)
   → Matches themes like: emotional sci-fi like Interstellar

🎬 Inception (2010) (⭐ 4.07)
   → Matches themes like: emotional sci-fi like Interstellar



In [ ]:
while True:
    q = input("\nEnter your mood: ")

    if q.lower() == "exit":
        break

    recs = recommend_movies(q)
    explain(q, recs)


Enter your mood: Anxious

🎯 Based on: 'Anxious'

🎬 Power of Nightmares, The: The Rise of the Politics of Fear (2004) (⭐ 4.5)
   → Matches themes like: Anxious

🎬 Fantastic Fear of Everything, A (2012) (⭐ 4.0)
   → Matches themes like: Anxious

🎬 Panic (2000) (⭐ 4.0)
   → Matches themes like: Anxious

🎬 Pursuit of Happyness, The (2006) (⭐ 3.79)
   → Matches themes like: Anxious

🎬 Short Term 12 (2013) (⭐ 3.75)
   → Matches themes like: Anxious


Enter your mood: Exit
